In [0]:
# Gold Layer: Aggregate and summarize Silver layer joins
from pyspark.sql import functions as F

gold_schema = "salesproject.gold_layer"
silver_schema = "salesproject.silver_layer"

# 1. Customer Spending: total and average transaction amount per customer
customer_transactions = spark.table(f"{silver_schema}.customer_transactions")
customer_spending = customer_transactions.groupBy("customer_id").agg(
    F.sum("transaction_amount").alias("total_spent"),
    F.avg("transaction_amount").alias("avg_spent"),
    F.count("transaction_id").alias("num_transactions")
)
customer_spending.write.format("delta").mode("overwrite").saveAsTable(f"{gold_schema}.customer_spending")
print(f"Created {gold_schema}.customer_spending")

display(customer_spending.limit(5))

# 2. Account Performance: opportunity metrics per account
account_opportunities = spark.table(f"{silver_schema}.account_opportunities")
account_performance = account_opportunities.groupBy("account_id").agg(
    F.count("opportunity_id").alias("num_opportunities"),
    F.sum("amount").alias("total_opportunity_value"),
    F.avg("amount").alias("avg_opportunity_value")
)
account_performance.write.format("delta").mode("overwrite").saveAsTable(f"{gold_schema}.account_performance")
print(f"Created {gold_schema}.account_performance")

display(account_performance.limit(5))

# 3. Sales Pipeline Summary: opportunity stages and win rates
sales_pipeline_summary = account_opportunities.groupBy("stage").agg(
    F.count("opportunity_id").alias("num_opportunities"),
    F.sum(F.when(F.col("stage") == "Closed Won", 1).otherwise(0)).alias("num_won"),
    F.avg("amount").alias("avg_opportunity_value")
)
sales_pipeline_summary.write.format("delta").mode("overwrite").saveAsTable(f"{gold_schema}.sales_pipeline_summary")
print(f"Created {gold_schema}.sales_pipeline_summary")

display(sales_pipeline_summary.limit(5))

# 4. Business Summary: overall KPIs
business_summary = customer_spending.agg(
    F.sum("total_spent").alias("total_customer_spending"),
    F.avg("avg_spent").alias("avg_customer_spending")
).crossJoin(
    account_performance.agg(
        F.sum("total_opportunity_value").alias("total_opportunity_value"),
        F.avg("avg_opportunity_value").alias("avg_account_opportunity")
    )
)
business_summary.write.format("delta").mode("overwrite").saveAsTable(f"{gold_schema}.business_summary")
print(f"Created {gold_schema}.business_summary")

display(business_summary)